In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in sorted(files):
        print(os.path.join(root, f))

/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv
/kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv


In [2]:
import os
import pandas as pd

train_path = None
test_path = None
sub_path = None

for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        full = os.path.join(root, f)
        if f.startswith("train.csv"):
            train_path = full
        elif f.startswith("test.csv"):
            test_path = full
        elif f.startswith("sample_submission.csv"):
            sub_path = full

print("train:", train_path)
print("test :", test_path)
print("sub  :", sub_path)

assert train_path is not None, "train file not found under /kaggle/input — check the Input panel!"
train_df = pd.read_csv(train_path, low_memory=False)
test_df = pd.read_csv(test_path, low_memory=False)
sub_df = pd.read_csv(sub_path, low_memory=False)
print("Loaded:", train_df.shape, test_df.shape)

train: /kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/train.csv
test : /kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/test.csv
sub  : /kaggle/input/competitions/workstation-rental-price-tier-prediction/kaggle_ready_dataset/sample_submission.csv
Loaded: (279746, 60) (114538, 59)


In [3]:
import glob
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import HistGradientBoostingClassifier

t0 = time.time()

# 1. Load data directly
input_files = glob.glob("/kaggle/input/**/*", recursive=True)
train_path = next((p for p in input_files if "train" in os.path.basename(p)), "train.csv.gz")
test_path  = next((p for p in input_files if "test" in os.path.basename(p)), "test.csv")
sub_path   = next((p for p in input_files if "sample_submission" in os.path.basename(p)), "sample_submission.csv")

train_df = pd.read_csv(train_path, low_memory=False)
test_df  = pd.read_csv(test_path, low_memory=False)
sub_df   = pd.read_csv(sub_path, low_memory=False)
print(f"[1] Loaded in {time.time() - t0:.1f}s")

# 2. Fast vectorized feature engineering (no slow per-row loops)
def fast_fe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.drop(columns=["price_tier"], errors="ignore").copy()

    if "facilities" in out.columns:
        s = out["facilities"].fillna("").astype(str)
        out["fac_cnt"] = np.where(s.str.len() > 2, s.str.count(",") + 1, 0).astype("float32")
        for fc in ["FCL-087", "FCL-078", "FCL-135", "FCL-056", "FCL-080", "FCL-058", "FCL-136", "FCL-077"]:
            out[f"f_{fc}"] = s.str.contains(fc, regex=False).astype("int8")
        out = out.drop(columns=["facilities"])

    for dcol in ["snapshot_start_date", "snapshot_end_date", "first_review_date", "last_review_date"]:
        if dcol in out.columns:
            s_d = out[dcol].astype(str)
            out[f"{dcol}_yr"] = pd.to_numeric(s_d.str.slice(0, 4), errors="coerce").astype("float32")
            out[f"{dcol}_mo"] = pd.to_numeric(s_d.str.slice(5, 7), errors="coerce").astype("float32")
            out = out.drop(columns=[dcol])

    out["seats_per_ws"] = (out["max_seats"] / (out["workstations"].fillna(0) + 1.0)).astype("float32")
    out["load_per_seat"] = (out["booked_load"] / (out["max_seats"] + 1.0)).astype("float32")
    out["metro_access"] = out["metro"].astype(str) + "_" + out["access_type"].astype(str)

    for col, cap in [("max_term_days", 3650.0), ("max_term_avg", 3650.0), ("reviews_per_month", 30.0)]:
        if col in out.columns:
            out[col] = np.clip(out[col].astype("float64"), 0.0, cap).astype("float32")

    return out

t1 = time.time()
y = train_df["price_tier"].astype(np.int8)
X_tr = fast_fe(train_df)
X_te = fast_fe(test_df)

both = pd.concat([X_tr, X_te], axis=0, ignore_index=True)
for gcol in ["operator_ref", "zone", "space_class", "metro", "borough", "metro_access"]:
    freq = both[gcol].value_counts(normalize=True)
    X_tr[f"{gcol}_freq"] = X_tr[gcol].map(freq).fillna(0.0).astype("float32")
    X_te[f"{gcol}_freq"] = X_te[gcol].map(freq).fillna(0.0).astype("float32")

for c in X_tr.select_dtypes(include="object").columns:
    vals = pd.concat([X_tr[c], X_te[c]], axis=0).fillna("MISSING").astype(str)
    codes = {v: i for i, v in enumerate(vals.unique())}
    X_tr[c] = X_tr[c].fillna("MISSING").astype(str).map(codes).astype("int32")
    X_te[c] = X_te[c].fillna("MISSING").astype(str).map(codes).astype("int32")

print(f"[2] Features ready in {time.time() - t1:.1f}s | cols={X_tr.shape[1]}")

# 3. 2-Fold CV with early stopping for fast iteration
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
oof = np.zeros(len(X_tr), dtype=np.int8)
test_probas = np.zeros((len(X_te), 4), dtype=np.float64)

for f, (tr_i, va_i) in enumerate(skf.split(X_tr, y)):
    tf = time.time()
    m = HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=180,
        max_leaf_nodes=63,
        min_samples_leaf=30,
        l2_regularization=1.5,
        early_stopping=True,
        n_iter_no_change=12,
        random_state=42 + f,
    )
    m.fit(X_tr.iloc[tr_i], y.iloc[tr_i])
    oof[va_i] = m.predict(X_tr.iloc[va_i])
    test_probas += m.predict_proba(X_te) / 2.0
    print(f"[3] Fold {f + 1} acc={accuracy_score(y.iloc[va_i], oof[va_i]):.5f} in {time.time() - tf:.1f}s")

print(f"2-Fold OOF acc={accuracy_score(y, oof):.5f} | Total={time.time() - t0:.1f}s")

# 4. Save submission.csv
sub = pd.DataFrame({
    "ID": sub_df["ID"],
    "price_tier": (test_probas.argmax(axis=1) + 1).astype(int),
})
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv:", sub.shape)

[1] Loaded in 6.1s
[2] Features ready in 9.0s | cols=80
[3] Fold 1 acc=0.71039 in 60.9s
[3] Fold 2 acc=0.71045 in 59.9s
2-Fold OOF acc=0.71042 | Total=135.9s
Saved submission.csv: (114538, 2)
